# DP2 – Handling missing time-series data with imputation

**Data:** `ex22PM10imputation.csv` – PM10 (fine dust) values measured in Salzburg-Lehen for one year, every **30 minutes**
(→ 48 values per day, 17 520 values in total).
- Column 1: time series with **300 missing values** (`NaN`)
- Column 2: the same time series **complete** (the real values – we use them only to check our results)

**Imputation** = filling in missing values with a good guess.

We try 4 methods:
| | method | idea |
|---|---|---|
| (a) | mean | replace by the mean of the whole series |
| (b) | last value | replace by the value just before (*extrapolation*) |
| (c) | linear interpolation | average of the value before and the value after |
| (d) | day before | replace by the value 24 h earlier (48 steps back) |

Hint from the exercise: missing values never come one after another, so the neighbours of a missing value are always known.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load the data (separator is `;`)

In [ ]:
df = pd.read_csv("../ex22PM10imputation.csv", sep=";", header=None, names=["pm10_missing", "pm10_true"])
print(df.head())
print("Rows:", len(df))
print("Missing values:", df["pm10_missing"].isna().sum())

missing = df["pm10_missing"].isna()      # True where a value is missing
idx = np.where(missing)[0]               # positions of the missing values
print("First missing positions:", idx[:10])

   pm10_missing  pm10_true
0         642.2      642.2
1         336.1      336.1
2         242.7      242.7
3         201.1      201.1
4         165.7      165.7
Rows: 17520
Missing values: 300
First missing positions: [ 99 157 215 273 331 389 447 504 562 620]


## 2. (a) Mean of the whole time series
Very simple and works for any data, but it ignores time completely – every missing value gets the same number.

In [ ]:
s = df["pm10_missing"]
imp_mean = s.fillna(s.mean())     # mean() ignores NaN automatically
print("Mean value used:", round(s.mean(), 2))

Mean value used: 20.86


## 3. (b) Last known value (forward fill)
We take the value directly before the gap. `ffill` = "forward fill".

In [ ]:
imp_last = s.ffill()

## 4. (c) Linear interpolation
We draw a straight line between the value before and the value after.
Because only one value is missing at a time, this is simply: `(before + after) / 2`.

In [ ]:
imp_linear = s.interpolate(method="linear")

## 5. (d) Value of the day before at the same time
One day = 48 steps, so we take the value 48 positions earlier: `shift(48)`.
(If that value is also missing or does not exist – e.g. on the first day – we use the last value as a fallback.)

In [ ]:
day_before = s.shift(48)
imp_day = s.fillna(day_before)       # fill NaN with the value 48 steps earlier
imp_day = imp_day.fillna(s.ffill())  # fallback for the few cases where that is also missing
print("Still missing:", imp_day.isna().sum())

Still missing: 0


## 6. Quick look at the results
We compare the imputed values with the true values at the 300 missing positions.
The error measure is the **mean absolute error (MAE)** = average of |imputed − true|.

In [ ]:
true = df["pm10_true"][missing]
results = pd.DataFrame({
    "true":       true,
    "mean":       imp_mean[missing],
    "last value": imp_last[missing],
    "linear":     imp_linear[missing],
    "day before": imp_day[missing],
})
print(results.head(10).round(1))

for m in ["mean", "last value", "linear", "day before"]:
    mae = (results[m] - results["true"]).abs().mean()
    print(f"MAE {m:12s}: {mae:.2f}")

     true  mean  last value  linear  day before
99   15.2  20.9        16.2    14.7        32.4
157   4.5  20.9         5.2     5.0         9.4
215   3.5  20.9         3.4     3.4        10.3
273   4.7  20.9         7.8     6.7         3.3
331  18.1  20.9        16.8    18.0         6.8
389  10.7  20.9        11.3    14.1        14.0
447  15.8  20.9        14.9    16.8        37.3
504  20.4  20.9        15.0    15.8         7.0
562  33.7  20.9        31.4    35.0         8.5
620  26.1  20.9        22.5    25.4        40.0
MAE mean        : 11.82
MAE last value  : 2.11
MAE linear      : 1.36
MAE day before  : 9.85


➡ Linear interpolation has the smallest error, the mean method the largest.
We save the results so DP3 and DP4 can use them.

In [ ]:
results.to_csv("imputation_results.csv")
print("saved imputation_results.csv")

saved imputation_results.csv


## Summary
- **Mean:** ignores time → large errors.
- **Last value:** good, because PM10 changes slowly within 30 min.
- **Linear interpolation:** best, uses information from *both* sides of the gap.
- **Day before:** worse – 24 h is a long time; weather changes from day to day.

The detailed evaluation follows in **DP3**.